# Utils - QB - PeriodPositionNormalizer

Ce notebook illustre et vérifie le comportement de la classe `PeriodPositionNormalizer`
(`tsforecast/utils/position/normalizer.py`), qui centralise la normalisation des
représentations de **position de période** : codes (`'S'`/`'E'`) vs noms littéraux
(`'start'`/`'end'`), ainsi que l'extraction/décomposition/recomposition de la
position à partir d'un offset pandas (`MS`, `ME`, `QS`, `QE`, etc.).

**Périmètre** : conformément à la consigne, seules les **méthodes publiques** de
`PeriodPositionNormalizer`, telles que définies dans `normalizer.py`, sont testées :
- `normalize(value)`
- `to_code(position)`
- `to_literal(position)`
- `validate(value)`
- `extract_position_from_offset(offset_str)`
- `decompose_offset(offset_str)`
- `combine_frequency_position(frequency, position)`
- `flip_position(position)`

`normalizer.py` ne définit aucune fonction libre (uniquement la classe). Les
méthodes privées (`__init__`) ne sont pas testées directement, mais les
mappings internes (`_code_to_literal`, `_literal_to_code`,
`_position_aware_frequencies`, `_legacy_offset_mapping`) sont **inspectés en
lecture seule** en section 1 pour servir de source de vérité aux tests qui
suivent — sans jamais en dépendre pour appeler une méthode privée.

**Remarque sur le périmètre** : le sous-module `tsforecast/utils/position/`
expose également, via `position/utils.py` (et non `normalizer.py`), des
fonctions libres (`normalize_position`, `to_literal`, `to_code`,
`validate_position`, `extract_position_from_offset`, `decompose_offset`,
`combine_frequency_position`, `flip_position`) qui enveloppent
`PeriodPositionNormalizer`. Comme pour `period_position_converter.ipynb`, ce
notebook se limite strictement à la classe définie dans `normalizer.py` ; ces
fonctions libres ne sont pas testées ici.

**Jeux de données** : `PeriodPositionNormalizer` opère sur des chaînes
(codes/littéraux de position, offsets pandas), pas directement sur des séries
temporelles. On réutilise malgré tout `df_timeseries`/`df_panel`, créés dans
`3 - QB - Panel a frequences mixtes heterogene.ipynb`, comme source
d'offsets et de fréquences *réalistes* (section 11) : fréquence globale de
l'index, fréquence de publication hétérogène de `depenses_publiques_pib`
selon le pays, etc. On les complète par des chaînes synthétiques ciblées
(section 2.2) pour couvrir les cas limites que ces jeux ne couvrent pas
naturellement : casse, multiplicateurs (`'2MS'`), offsets legacy (`'A'`,
`'AS'`, `'AE'`), offsets ancrés (`'W-MON'`), offsets/fréquences inconnus, et
types non-string.

**Remarque** : un premier notebook d'exploration plus succinct existait dans
`0 - QB - Utils.ipynb` (section 6.1) ; il couvrait `normalize()`,
`decompose_offset()` et `combine_frequency_position()` sur quelques cas
simples. Les classes ayant depuis été refactorées, il a surtout servi à
confirmer l'existence de ces méthodes et leur signature approximative.

Ce notebook a vocation à servir de base à de futurs tests unitaires
(`tests/utils/position/test_normalizer.py`, qui n'existe pas encore).

## 1 - Import et instanciation

In [ ]:
# Importation des modules
import warnings
from typing import get_args

import numpy as np
import pandas as pd

# Classe testée
from tsforecast.utils.position.normalizer import PeriodPositionNormalizer, PositionType, UserPositionType

# Configuration de l'affichage
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

# Instanciation du normalizer
normalizer = PeriodPositionNormalizer()

print("Positions (codes) déclarées dans le type PositionType :")
print(get_args(PositionType))
print()
print("Positions (littéraux) déclarées dans le type UserPositionType :")
print(get_args(UserPositionType))

Inspection en lecture seule des mappings internes construits dans `__init__` :
ils servent de source de vérité pour les tests qui suivent (notamment pour
`combine_frequency_position()`, où la liste `_position_aware_frequencies`
joue un rôle central), sans qu'on appelle jamais de méthode privée.

In [ ]:
# Mappings réellement utilisés par la classe (source de vérité)
print("_code_to_literal :", normalizer._code_to_literal)
print("_literal_to_code :", normalizer._literal_to_code)
print()
print("_position_aware_frequencies (fréquences qui acceptent un suffixe S/E) :")
print(normalizer._position_aware_frequencies)
print()
print("_legacy_offset_mapping (offset pandas -> (fréquence, position)) :")
for offset, (freq, pos) in normalizer._legacy_offset_mapping.items():
    print(f"  {offset:4s} -> freq={freq!r:4s} position={pos!r}")

**Point de vigilance n°0**, visible dès l'inspection des mappings : `'A'`
(alias legacy de `'Y'`, annuel) est bien présent dans `_legacy_offset_mapping`
(avec `'AS'`/`'AE'`), mais **absent** de `_position_aware_frequencies` (qui ne
contient que `['M', 'Q', 'Y', 'W', 'B']`, pas `'A'`). Les deux mappings, qui
se recoupent pourtant conceptuellement, sont incohérents l'un avec l'autre —
conséquence directe en section 9.

## 2 - Jeux de données

### 2.1 - Reprise des jeux de données de `3 - QB - Panel a frequences mixtes heterogene.ipynb`

Les deux fonctions génératrices sont recopiées telles quelles (aucune fonction
partagée n'existe entre notebooks dans ce projet) pour obtenir `df_timeseries`
(séries temporelles macroéconomiques) et `df_panel` (panel France/Allemagne/Italie
à couverture et fréquences hétérogènes). Elles ne sont pas passées directement
à `PeriodPositionNormalizer` (qui opère sur des chaînes), mais servent de
contexte réaliste en section 11 : fréquences d'index détectées, fréquence de
publication hétérogène de `depenses_publiques_pib` selon le pays, etc.

In [ ]:
# Fonction de création de séries temporelles (recopiée depuis le notebook 3)
def create_timeseries_dataset(
    start_date: str = '2018-01-01',
    end_date: str = '2024-07-01',
    seed: int = 42
) -> pd.DataFrame:
    """Create a realistic macroeconomic time series dataset with mixed frequencies.

    Args:
        start_date: Start date for the dataset.
        end_date: End date for the dataset.
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with mixed-frequency macroeconomic indicators.
    """
    np.random.seed(seed)

    dates = pd.date_range(start=start_date, end=end_date, freq='MS')
    n_periods = len(dates)

    df = pd.DataFrame(index=dates)
    df.index.name = 'date'

    # Production industrielle (mensuelle, croissance avec bruit)
    trend = np.linspace(100, 115, n_periods)
    seasonal = 3 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
    noise = np.random.normal(0, 1.5, n_periods)
    df['production_industrielle'] = trend + seasonal + noise

    # Inflation mensuelle (IPC)
    inflation_trend = np.linspace(1.2, 2.8, n_periods)
    inflation_noise = np.random.normal(0, 0.3, n_periods)
    df['inflation_ipc'] = np.clip(inflation_trend + inflation_noise, 0.5, 5.0)

    # Taux de chômage (mensuel)
    chomage_trend = np.concatenate([
        np.linspace(8.5, 7.0, n_periods // 3),
        np.linspace(7.0, 9.5, n_periods // 3),
        np.linspace(9.5, 7.5, n_periods - 2 * (n_periods // 3))
    ])
    chomage_noise = np.random.normal(0, 0.2, n_periods)
    df['taux_chomage'] = np.clip(chomage_trend + chomage_noise, 4.0, 15.0)

    # PIB trimestriel
    pib_base = 2500
    pib_growth_quarterly = 0.5
    df['pib_trimestriel'] = np.nan
    quarter_start_months = [1, 4, 7, 10]
    quarter_idx = 0
    for i, date in enumerate(dates):
        if date.month in quarter_start_months:
            growth = pib_growth_quarterly + np.random.normal(0, 0.3)
            df.loc[date, 'pib_trimestriel'] = pib_base * (1 + growth / 100) ** quarter_idx
            quarter_idx += 1

    # Balance commerciale annuelle
    df['balance_commerciale_annuelle'] = np.nan
    for i, date in enumerate(dates):
        if date.month == 1:
            year_factor = (date.year - 2018)
            base_balance = -25 + year_factor * 3 + np.random.normal(0, 5)
            df.loc[date, 'balance_commerciale_annuelle'] = base_balance

    # Simulation des délais de publication
    df.loc[df.index[-1], 'inflation_ipc'] = np.nan
    df.loc[df.index[-1], 'taux_chomage'] = np.nan

    pib_available = df[df['pib_trimestriel'].notna()].index
    if len(pib_available) > 0:
        df.loc[pib_available[-1], 'pib_trimestriel'] = np.nan

    bc_available = df[df['balance_commerciale_annuelle'].notna()].index
    if len(bc_available) > 0:
        df.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan

    # Simulation de données historiques limitées
    mask_before_2019 = df.index < '2019-01-01'
    df.loc[mask_before_2019, 'production_industrielle'] = np.nan

    return df


df_timeseries = create_timeseries_dataset()
print(f"df_timeseries : {df_timeseries.shape}, {df_timeseries.index.min().date()} -> {df_timeseries.index.max().date()}")
df_timeseries.tail()

In [ ]:
# Fonction de création d'un jeu de données de panel (recopiée depuis le notebook 3)
def create_panel_dataset(seed: int = 42) -> pd.DataFrame:
    """Create a realistic macroeconomic panel dataset with mixed frequencies.

    Each entity has its own coverage period (start/end dates) and its own
    publication frequency for the public spending indicator, to simulate a
    heterogeneous panel across entities.

    Args:
        seed: Random seed for reproducibility.

    Returns:
        DataFrame with MultiIndex (country, date) and mixed-frequency indicators.
    """
    np.random.seed(seed)

    countries = {
        'France': {
            'pib_base': 2800, 'inflation_base': 1.5, 'chomage_base': 8.0, 'depenses_base': 55.0,
            'start_date': '2018-01-01', 'end_date': '2024-07-01', 'prod_ind_start': '2018-06-01',
            'depenses_frequency': 'annuelle'
        },
        'Allemagne': {
            'pib_base': 3500, 'inflation_base': 1.2, 'chomage_base': 5.5, 'depenses_base': 45.0,
            'start_date': '2018-07-01', 'end_date': '2024-04-01', 'prod_ind_start': '2019-01-01',
            'depenses_frequency': 'trimestrielle'
        },
        'Italie': {
            'pib_base': 2200, 'inflation_base': 1.8, 'chomage_base': 10.5, 'depenses_base': 50.0,
            'start_date': '2019-01-01', 'end_date': '2024-07-01', 'prod_ind_start': '2019-06-01',
            'depenses_frequency': 'annuelle'
        }
    }

    all_data = []
    for country, params in countries.items():
        np.random.seed(seed + hash(country) % 1000)

        dates = pd.date_range(start=params['start_date'], end=params['end_date'], freq='MS')
        n_periods = len(dates)

        df_country = pd.DataFrame(index=dates)
        df_country['country'] = country

        # Production industrielle
        trend = np.linspace(100, 112 + np.random.uniform(-3, 3), n_periods)
        seasonal = 2.5 * np.sin(2 * np.pi * np.arange(n_periods) / 12)
        noise = np.random.normal(0, 1.2, n_periods)
        df_country['production_industrielle'] = trend + seasonal + noise
        prod_start = pd.Timestamp(params['prod_ind_start'])
        df_country.loc[df_country.index < prod_start, 'production_industrielle'] = np.nan

        # Inflation
        infl_trend = np.linspace(
            params['inflation_base'],
            params['inflation_base'] + np.random.uniform(0.5, 2.0),
            n_periods
        )
        infl_noise = np.random.normal(0, 0.25, n_periods)
        df_country['inflation_ipc'] = np.clip(infl_trend + infl_noise, 0.3, 6.0)

        # Taux de chômage
        chomage_base = params['chomage_base']
        chomage_evolution = np.concatenate([
            np.linspace(chomage_base, chomage_base - 1, n_periods // 3),
            np.linspace(chomage_base - 1, chomage_base + 2, n_periods // 3),
            np.linspace(chomage_base + 2, chomage_base + 0.5, n_periods - 2 * (n_periods // 3))
        ])
        chomage_noise = np.random.normal(0, 0.15, n_periods)
        df_country['taux_chomage'] = np.clip(chomage_evolution + chomage_noise, 2.5, 15.0)

        # PIB trimestriel
        df_country['pib_trimestriel'] = np.nan
        quarter_end_months = [1, 4, 7, 10]
        quarter_idx = 0
        for date in dates:
            if date.month in quarter_end_months:
                growth = 0.4 + np.random.normal(0, 0.35)
                df_country.loc[date, 'pib_trimestriel'] = params['pib_base'] * (1 + growth / 100) ** quarter_idx
                quarter_idx += 1

        # Balance commerciale annuelle
        df_country['balance_commerciale_annuelle'] = np.nan
        for date in dates:
            if date.month == 1:
                year_factor = (date.year - 2018)
                base = -20 + np.random.uniform(-10, 10) + year_factor * 2
                df_country.loc[date, 'balance_commerciale_annuelle'] = base

        # Dépenses publiques : fréquence annuelle ou trimestrielle selon le pays
        df_country['depenses_publiques_pib'] = np.nan
        publication_months = [1] if params['depenses_frequency'] == 'annuelle' else [1, 4, 7, 10]
        depenses_idx = 0
        for date in dates:
            if date.month in publication_months:
                value = params['depenses_base'] + 0.1 * depenses_idx + np.random.normal(0, 1.0)
                df_country.loc[date, 'depenses_publiques_pib'] = value
                depenses_idx += 1

        # Délais de publication
        df_country.loc[df_country.index[-1], 'inflation_ipc'] = np.nan
        df_country.loc[df_country.index[-1], 'taux_chomage'] = np.nan

        pib_available = df_country[df_country['pib_trimestriel'].notna()].index
        if len(pib_available) > 0:
            df_country.loc[pib_available[-1], 'pib_trimestriel'] = np.nan
        bc_available = df_country[df_country['balance_commerciale_annuelle'].notna()].index
        if len(bc_available) > 0:
            df_country.loc[bc_available[-1], 'balance_commerciale_annuelle'] = np.nan
        depenses_available = df_country[df_country['depenses_publiques_pib'].notna()].index
        if len(depenses_available) > 0:
            df_country.loc[depenses_available[-1], 'depenses_publiques_pib'] = np.nan

        all_data.append(df_country)

    df_panel = pd.concat(all_data, ignore_index=False)
    df_panel = df_panel.reset_index().rename(columns={'index': 'date'})
    df_panel = df_panel.set_index(['country', 'date'])
    df_panel = df_panel.sort_index()

    return df_panel


df_panel = create_panel_dataset()
print(f"df_panel : {df_panel.shape}, entités : {df_panel.index.get_level_values('country').unique().tolist()}")
for country in df_panel.index.get_level_values('country').unique():
    dates_country = df_panel.loc[country].index
    print(f"  {country:12s} {dates_country.min().strftime('%Y-%m')} -> {dates_country.max().strftime('%Y-%m')}")
df_panel.loc['France'].tail()

### 2.2 - Chaînes synthétiques ciblées

`PeriodPositionNormalizer` opère sur des chaînes, pas sur des `Series`/`DataFrame` :
l'essentiel des cas limites se couvre donc avec des chaînes synthétiques plutôt
qu'avec `df_timeseries`/`df_panel`. On couvre ici : codes/littéraux valides,
variantes de casse, types non-string, offsets connus (avec et sans
multiplicateur), offsets legacy (`A`/`AS`/`AE`), offsets ancrés, offsets/fréquences
inconnus, et fréquences non "position-aware".

In [ ]:
# Positions valides, sous toutes leurs formes
positions_codes = ['S', 'E']
positions_litterales = ['start', 'end']

# Variantes de casse et quasi-valides : AUCUNE n'est supportée (seules 'S'/'E'
# et 'start'/'end' exactement le sont)
positions_casse_invalide = ['s', 'e', 'Start', 'START', 'End', 'End'.upper()]

# Types non-string : passage direct dans normalize() (vérification de type explicite)
positions_types_invalides = [None, 1, ['S'], ('start',), np.nan]

# Positions strings mais sémantiquement invalides
positions_invalides = ['', 'middle', 'MS', 'debut', 'fin']

# Offsets pandas connus (avec position explicite), un par fréquence "position-aware"
offsets_connus = ['MS', 'ME', 'M', 'QS', 'QE', 'Q', 'YS', 'YE', 'Y', 'WS', 'WE', 'W', 'BS', 'BE', 'B']

# Offsets legacy annuels : alias de Y, présents dans _legacy_offset_mapping
offsets_legacy_annuels = ['AS', 'AE', 'A']

# Offsets avec multiplicateur pandas (ex. '2MS' = tous les 2 mois, début de mois)
offsets_avec_multiplicateur = ['2MS', '3QE', '12M']

# Offsets ancrés / inconnus / non position-aware
offsets_ancres_inconnus = ['W-MON', 'D', 'H', 'min', 'SM', 'foo', '']

# Fréquences (bases, sans position) : position-aware vs non position-aware
#frequences_position_aware = normalizer._position_aware_frequencies  # ['M', 'Q', 'Y', 'W', 'B']
frequences_non_position_aware = ['D', 'H', 'T', 'min', 'A']  # 'A' inclus : cf. point de vigilance n°0

print(f"positions_codes                : {positions_codes}")
print(f"positions_litterales           : {positions_litterales}")
print(f"positions_casse_invalide       : {positions_casse_invalide}")
print(f"positions_types_invalides      : {positions_types_invalides}")
print(f"positions_invalides            : {positions_invalides}")
print(f"offsets_connus                 : {offsets_connus}")
print(f"offsets_legacy_annuels         : {offsets_legacy_annuels}")
print(f"offsets_avec_multiplicateur    : {offsets_avec_multiplicateur}")
print(f"offsets_ancres_inconnus        : {offsets_ancres_inconnus}")
print(f"frequences_position_aware      : {frequences_position_aware}")
print(f"frequences_non_position_aware  : {frequences_non_position_aware}")

## 3 - `normalize()` : normalisation vers un code

Convertit n'importe quelle représentation supportée (code ou littéral) vers un
code (`'S'`/`'E'`). C'est la méthode pivot : toutes les autres méthodes
publiques (sauf `extract_position_from_offset`/`decompose_offset`/
`combine_frequency_position`, qui opèrent sur des offsets) s'appuient dessus.

In [ ]:
# Un code supporté est toujours renvoyé inchangé
for code_valide in positions_codes:
    print(f"normalize({code_valide!r}) -> {normalizer.normalize(code_valide)!r}")

# Chaque nom littéral supporté est converti vers son code correspondant
for litteral_valide in positions_litterales:
    print(f"normalize({litteral_valide!r}) -> {normalizer.normalize(litteral_valide)!r}")

**Point de vigilance n°1** : la normalisation est **strictement sensible à la
casse**, et n'accepte aucune variante autre que les 4 formes exactes déclarées
dans les mappings (`'S'`, `'E'`, `'start'`, `'end'`) — y compris des chaînes
qui semblent raisonnables au premier abord (`'Start'`, `'s'`, `'MS'` n'est
d'ailleurs pas une *position* mais un *offset*, donc logiquement rejeté ici).

In [ ]:
# Casse et quasi-valides : toutes ces variantes échouent
for v in positions_casse_invalide + positions_invalides:
    try:
        r = normalizer.normalize(v)
        print(f"normalize({v!r}) -> {r!r} (INATTENDU : pas d'erreur)")
    except ValueError as e:
        print(f"normalize({v!r}) -> ValueError : {e}")

**Point de vigilance n°2** : `normalize()` vérifie explicitement le type
(`isinstance(value, str)`) et lève une `ValueError` **dédiée** (pas de
`TypeError`) pour tout type non-string — contrairement à
`extract_position_from_offset()`/`decompose_offset()`, qui ne font aucune
vérification de type et laissent remonter une `TypeError` native de `re`
(cf. section 7/8).

In [ ]:
# Types non-string : ValueError dédiée (message différent de celui du cas "string invalide")
for v in positions_types_invalides:
    try:
        r = normalizer.normalize(v)
        print(f"normalize({v!r}) -> {r!r} (INATTENDU : pas d'erreur)")
    except ValueError as e:
        print(f"normalize({v!r}) -> ValueError : {e}")

## 4 - `to_code()` : alias explicite de `normalize()`

D'après la docstring, `to_code()` se contente de déléguer à `normalize()`.
Vérification empirique sur l'ensemble des cas déjà couverts en section 3 :
mêmes résultats, mêmes exceptions.

In [ ]:
# to_code() doit produire EXACTEMENT le même résultat (ou la même exception) que normalize()
tous_les_cas = (
    positions_codes + positions_litterales + positions_casse_invalide
    + positions_invalides + positions_types_invalides
)

for v in tous_les_cas:
    try:
        r_normalize = normalizer.normalize(v)
        erreur_normalize = None
    except ValueError as e:
        r_normalize, erreur_normalize = None, str(e)

    try:
        r_to_code = normalizer.to_code(v)
        erreur_to_code = None
    except ValueError as e:
        r_to_code, erreur_to_code = None, str(e)

    assert r_normalize == r_to_code
    assert erreur_normalize == erreur_to_code

print(f"OK : to_code() == normalize() (résultat ET message d'erreur) sur les {len(tous_les_cas)} cas testés")

## 5 - `to_literal()` : conversion vers le nom littéral

Normalise d'abord (via `normalize()`), puis convertit le code obtenu en son
nom littéral via `_code_to_literal`.

In [ ]:
# Code -> littéral et littéral -> littéral (idempotent : normalize() puis lookup)
for v in positions_codes + positions_litterales:
    print(f"to_literal({v!r}) -> {normalizer.to_literal(v)!r}")

# Round-trip complet : to_code(to_literal(x)) == normalize(x)
for v in positions_codes + positions_litterales:
    assert normalizer.to_code(normalizer.to_literal(v)) == normalizer.normalize(v)
print("\nOK : to_code(to_literal(x)) == normalize(x) pour tous les formats valides")

# Entrée invalide : l'erreur de normalize() est propagée telle quelle
try:
    normalizer.to_literal('middle')
except ValueError as e:
    print("\nto_literal('middle') -> ValueError :", e)

## 6 - `validate()` : vérification booléenne, ne lève jamais d'exception

Enveloppe `normalize()` dans un `try/except ValueError`. Doit donc toujours
renvoyer un booléen, y compris pour des entrées de types complètement
différents (contrairement à `normalize()`, qui lève sur les types non-string).

In [ ]:
# Valeurs valides -> True
for v in positions_codes + positions_litterales:
    print(f"validate({v!r}) -> {normalizer.validate(v)}")

print()
# Valeurs invalides de tous types -> toujours False, jamais d'exception
# (y compris les types non-string, capturés en interne via normalize())
for v in positions_casse_invalide + positions_invalides + positions_types_invalides:
    resultat = normalizer.validate(v)
    print(f"validate({v!r}) -> {resultat}")
    assert resultat is False

print("\nOK : validate() ne lève jamais, quel que soit le type ou la valeur passée")

## 7 - `extract_position_from_offset()` : extraction depuis un offset pandas

Nettoie l'offset de tout multiplicateur numérique en tête (`re.sub(r'^\d+',
'', offset_str)`), puis recherche l'offset nettoyé dans
`_legacy_offset_mapping`. Aucune vérification de type n'est faite en amont.

In [ ]:
# Offsets connus : extraction correcte de la position (y compris les formes
# sans suffixe explicite, qui retombent sur leur défaut -- 'E' pour M/Q/Y/W/B)
for offset in offsets_connus:
    print(f"extract_position_from_offset({offset!r}) -> {normalizer.extract_position_from_offset(offset)!r}")

print()
# Offsets legacy annuels : 'A' est bien reconnu comme alias de 'Y' ici (contrairement
# à combine_frequency_position(), cf. section 9)
for offset in offsets_legacy_annuels:
    print(f"extract_position_from_offset({offset!r}) -> {normalizer.extract_position_from_offset(offset)!r}")

In [ ]:
# Multiplicateur : correctement retiré avant recherche dans le mapping
for offset in offsets_avec_multiplicateur:
    print(f"extract_position_from_offset({offset!r}) -> {normalizer.extract_position_from_offset(offset)!r}")

**Point de vigilance n°3** : pour tout offset **absent** de
`_legacy_offset_mapping` (ancré comme `'W-MON'`, non position-aware comme
`'D'`/`'H'`, ou franchement inconnu comme `'foo'`), la méthode retombe
**silencieusement** sur la position par défaut `'E'`, sans jamais lever
d'erreur ni avertir. Un offset ancré comme `'W-MON'` n'est même pas partiellement
reconnu (le `'W'` initial n'est pas extrait séparément de l'ancre `'-MON'`) :
la chaîne entière est cherchée telle quelle dans le mapping, échoue, et
retombe sur `'E'`.

In [ ]:
# Offsets ancrés / inconnus / non position-aware : fallback silencieux sur 'E'
for offset in offsets_ancres_inconnus:
    r = normalizer.extract_position_from_offset(offset)
    print(f"extract_position_from_offset({offset!r}) -> {r!r}")
    assert r == 'E'
print("\nOK (mais discutable) : tous ces offsets non reconnus retombent sur 'E', sans erreur ni avertissement")

**Point de vigilance n°4** : aucune vérification de type n'étant faite,
passer une valeur non-string lève une `TypeError` **native** de `re.sub()`
(pas la `ValueError` dédiée que lève `normalize()` dans le même cas) —
incohérence à documenter pour les futurs tests unitaires.

In [ ]:
# Type non-string -> TypeError (pas ValueError), levée par re.sub() en interne
for v in [123, None, ['MS']]:
    try:
        normalizer.extract_position_from_offset(v)
        print(f"extract_position_from_offset({v!r}) -> PAS D'ERREUR (inattendu)")
    except TypeError as e:
        print(f"extract_position_from_offset({v!r}) -> TypeError : {e}")
    except ValueError as e:
        print(f"extract_position_from_offset({v!r}) -> ValueError (inattendu) : {e}")

## 8 - `decompose_offset()` : décomposition en (fréquence, position)

Sépare le multiplicateur numérique (`re.match(r'^(\d+)(.+)', ...)`) puis
recherche la base nettoyée dans `_legacy_offset_mapping`, qui donne
directement `(frequence, position)`. Le multiplicateur, une fois isolé, est
purement et simplement **jeté** (jamais réinjecté dans le tuple renvoyé).

In [ ]:
# Offsets connus : décomposition correcte en (fréquence, position)
for offset in offsets_connus + offsets_legacy_annuels:
    print(f"decompose_offset({offset!r}) -> {normalizer.decompose_offset(offset)}")

print()
# Multiplicateur : isolé puis PERDU -- seule la base (fréquence, position) est renvoyée
for offset in offsets_avec_multiplicateur:
    print(f"decompose_offset({offset!r}) -> {normalizer.decompose_offset(offset)} (multiplicateur perdu)")

**Point de vigilance n°5** : comme pour `extract_position_from_offset()`, tout
offset absent du mapping (`'D'`, `'foo'`, `'W-MON'`, ...) retombe
silencieusement sur `(clean_offset, 'E')` — y compris `'D'`, une fréquence
pandas parfaitement valide mais qui **n'a pas** de notion de position
(`'D'` n'est pas dans `_position_aware_frequencies`). `decompose_offset()` ne
consulte jamais cette liste : il affirme une position `'E'` pour `'D'` comme
il le ferait pour `'M'`, sans distinction.

In [ ]:
# Offsets non reconnus (y compris 'D', pourtant valide mais non position-aware) :
# fallback silencieux sur (offset_nettoye, 'E')
for offset in offsets_ancres_inconnus:
    r = normalizer.decompose_offset(offset)
    print(f"decompose_offset({offset!r}) -> {r}")
    assert r[1] == 'E'

# Cohérence interne : extract_position_from_offset() et decompose_offset()
# s'accordent toujours sur la position extraite (même logique de fallback)
for offset in offsets_connus + offsets_legacy_annuels + offsets_ancres_inconnus:
    assert normalizer.extract_position_from_offset(offset) == normalizer.decompose_offset(offset)[1]
print("\nOK : extract_position_from_offset(x) == decompose_offset(x)[1] pour tous les offsets testés")

In [ ]:
# Type non-string -> TypeError native de re.match(), même incohérence qu'en section 7
for v in [123, None]:
    try:
        normalizer.decompose_offset(v)
        print(f"decompose_offset({v!r}) -> PAS D'ERREUR (inattendu)")
    except TypeError as e:
        print(f"decompose_offset({v!r}) -> TypeError : {e}")

## 9 - `combine_frequency_position()` : recomposition fréquence + position

Opération "inverse" de `decompose_offset()`, mais **pas symétrique** : elle ne
consulte pas `_legacy_offset_mapping` mais `_position_aware_frequencies`, une
liste distincte et plus restreinte (`['M', 'Q', 'Y', 'W', 'B']`, sans `'A'`).

In [ ]:
# Fréquence position-aware + position fournie (code ou littéral) -> combinaison
for freq in frequences_position_aware:
    for position in ['S', 'end']:
        r = normalizer.combine_frequency_position(freq, position)
        print(f"combine_frequency_position({freq!r}, {position!r}) -> {r!r}")

**Point de vigilance n°6 (majeur)** : conséquence directe de l'asymétrie
notée en section 1 (point de vigilance n°0). `'A'` est un alias legacy
reconnu de `'Y'` par `decompose_offset()`/`extract_position_from_offset()`,
mais `combine_frequency_position()` ne le reconnaît **pas** comme
"position-aware" : la position demandée est silencieusement **ignorée**, sans
erreur ni avertissement, alors que la même position demandée avec `'Y'` (la
forme non-legacy équivalente) produit bien un suffixe.

In [ ]:
# 'A' (legacy) vs 'Y' (canonique) : même sémantique en théorie, comportement
# radicalement différent dans combine_frequency_position()
r_A = normalizer.combine_frequency_position('A', 'start')
r_Y = normalizer.combine_frequency_position('Y', 'start')
print(f"combine_frequency_position('A', 'start') -> {r_A!r}  (position ignorée, silencieusement)")
print(f"combine_frequency_position('Y', 'start') -> {r_Y!r}  (position appliquée)")
assert r_A == 'A' and r_Y == 'YS'
print("\nOK (mais incohérent) : 'A' et 'Y' ne se comportent PAS de la même façon malgré leur équivalence dans _legacy_offset_mapping")

Pour les fréquences **non** position-aware (hors `M`/`Q`/`Y`/`W`/`B`), la
méthode renvoie la fréquence inchangée quelle que soit la position demandée —
**sans avertissement**, contrairement au cas `position=None` avec une
fréquence position-aware, qui lui **avertit** (cf. cellule suivante).

In [ ]:
# Fréquences non position-aware : no-op silencieux, quelle que soit la position demandée
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    for freq in frequences_non_position_aware:
        for position in ['start', 'E', None]:
            r = normalizer.combine_frequency_position(freq, position)
            assert r == freq
    print(f"OK : {len(frequences_non_position_aware)} fréquences non position-aware x 3 positions -> toujours no-op")
    print(f"Nombre de warnings émis : {len(w)} (0 attendu, cf. cellule suivante pour le cas position=None + freq position-aware)")

In [ ]:
# position=None sur une fréquence position-aware -> UserWarning explicite,
# ET renvoie la fréquence SEULE (sans suffixe), comme si elle n'était pas position-aware
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    r = normalizer.combine_frequency_position('M', None)
    print(f"combine_frequency_position('M', None) -> {r!r}")
    print(f"Warning(s) émis : {len(w)}")
    if w:
        print(f"  {w[0].category.__name__} : {w[0].message}")
    assert r == 'M'
    assert len(w) == 1 and issubclass(w[0].category, UserWarning)

In [ ]:
# Position invalide avec une fréquence position-aware -> ValueError propagée depuis normalize()
try:
    normalizer.combine_frequency_position('M', 'middle')
except ValueError as e:
    print("combine_frequency_position('M', 'middle') -> ValueError :", e)

## 10 - `flip_position()` : inversion de position

Normalise d'abord la position d'entrée (code ou littéral accepté), puis
renvoie l'opposé **en code**, quel que soit le format d'entrée.

In [ ]:
# Codes et littéraux acceptés en entrée ; TOUJOURS un code en sortie, même
# si l'entrée était un littéral (pas de symétrie de format)
for v in positions_codes + positions_litterales:
    r = normalizer.flip_position(v)
    print(f"flip_position({v!r}) -> {r!r}")
    assert r in ('S', 'E')

print()
# Double flip = identité (sur le code normalisé)
for v in positions_codes + positions_litterales:
    assert normalizer.flip_position(normalizer.flip_position(v)) == normalizer.normalize(v)
print("OK : flip_position(flip_position(x)) == normalize(x)")

In [ ]:
# Position invalide -> ValueError propagée depuis normalize()
try:
    normalizer.flip_position('middle')
except ValueError as e:
    print("flip_position('middle') -> ValueError :", e)

## 11 - Application aux jeux de données réalistes (`df_timeseries` / `df_panel`)

`PeriodPositionNormalizer` n'agit jamais directement sur ces `DataFrame` (ce
rôle revient à `PeriodPositionConverter`, testé dans
`period_position_converter.ipynb`). On l'utilise ici pour raisonner sur les
**offsets/fréquences réels** qu'on peut en extraire, dans un scénario proche
de la mise en pratique du projet.

In [ ]:
# Fréquence globale de l'index de df_timeseries (détectée par pandas) :
# décomposition en (fréquence, position) via decompose_offset()
freq_index = df_timeseries.index.freqstr or pd.infer_freq(df_timeseries.index)
freq_base, position_index = normalizer.decompose_offset(freq_index)
print(f"Fréquence détectée sur df_timeseries.index : {freq_index!r}")
print(f"  -> decompose_offset({freq_index!r}) = (freq={freq_base!r}, position={position_index!r})")
print(f"  -> to_literal({position_index!r}) = {normalizer.to_literal(position_index)!r}")

In [ ]:
# pib_trimestriel : valeurs présentes uniquement aux mois de début de trimestre
# (index MS) -> on la modélise comme une variable trimestrielle en position
# 'start' via combine_frequency_position(), puis on vérifie le round-trip
offset_pib = normalizer.combine_frequency_position('Q', 'start')
print(f"Offset combiné pour pib_trimestriel (Q, start) -> {offset_pib!r}")

freq_pib, position_pib = normalizer.decompose_offset(offset_pib)
print(f"decompose_offset({offset_pib!r}) -> (freq={freq_pib!r}, position={position_pib!r})")
assert (freq_pib, position_pib) == ('Q', 'S')

# Si l'indicateur était plutôt publié en position de fin de trimestre (convention
# fréquente pour un flux constaté a posteriori), flip_position() donne l'offset opposé
offset_pib_fin = normalizer.combine_frequency_position('Q', normalizer.flip_position(position_pib))
print(f"Offset opposé (fin de trimestre) -> {offset_pib_fin!r}")

In [ ]:
# depenses_publiques_pib : fréquence de PUBLICATION hétérogène par pays dans
# df_panel (annuelle pour France/Italie, trimestrielle pour l'Allemagne) --
# on construit l'offset caractéristique de chaque pays
frequence_publication_par_pays = {'France': 'Y', 'Allemagne': 'Q', 'Italie': 'Y'}

for pays, freq in frequence_publication_par_pays.items():
    offset = normalizer.combine_frequency_position(freq, 'start')
    valeurs_disponibles = df_panel.loc[pays, 'depenses_publiques_pib'].dropna()
    print(f"{pays:10s} : fréquence de publication {freq!r} -> offset {offset!r} "
          f"({len(valeurs_disponibles)} valeurs publiées)")

In [ ]:
# balance_commerciale_annuelle : publiée en janvier (donc en position 'start'
# d'année dans l'index MS de df_timeseries) -> offset caractéristique de
# l'indicateur, indépendant de la fréquence MS de l'index global du DataFrame
offset_balance = normalizer.combine_frequency_position('Y', 'start')
freq_balance, position_balance = normalizer.decompose_offset(offset_balance)
print(f"balance_commerciale_annuelle : offset caractéristique {offset_balance!r}")
print(f"  -> position littérale : {normalizer.to_literal(position_balance)!r}")

# Comparaison avec la position de l'indice global (MS, donc 'start' également) :
# les deux coïncident ici, mais rien ne le garantit en général (d'où l'intérêt
# de traiter la position de CHAQUE indicateur indépendamment de l'index du DataFrame)
assert position_balance == position_index
print(f"  -> coïncide avec la position de l'index global ({position_index!r}), mais ce n'est pas garanti en général")

## 12 - Synthèse : pistes pour de futurs tests unitaires

Comportements observés dans ce notebook, à couvrir explicitement dans
`tests/utils/position/test_normalizer.py` :

- **`normalize()`** : strictement sensible à la casse, n'accepte que 4 formes
  exactes (`'S'`, `'E'`, `'start'`, `'end'`). Lève une `ValueError` dédiée
  (message explicite listant les formats supportés) aussi bien pour une
  chaîne non reconnue que pour un type non-string (vérification de type
  explicite en amont, contrairement aux méthodes liées aux offsets).
- **`to_code()`** : alias strict de `normalize()` — mêmes résultats, mêmes
  exceptions (message inclus), vérifié sur l'ensemble des cas valides et
  invalides.
- **`to_literal()`** : `normalize()` puis lookup ; round-trip
  `to_code(to_literal(x)) == normalize(x)` vérifié pour tous les formats
  valides. Propage la `ValueError` de `normalize()` telle quelle.
- **`validate()`** : ne lève **jamais**, quel que soit le type de l'entrée
  (contrairement à `normalize()`), toujours `True`/`False`.
- **`extract_position_from_offset()`** et **`decompose_offset()`** :
  - reconnaissent les offsets usuels (`MS`/`ME`/`QS`/`QE`/...) ainsi que les
    alias legacy annuels (`A`/`AS`/`AE`), avec ou sans multiplicateur
    numérique en tête (`'2MS'`) — le multiplicateur est systématiquement
    **perdu** dans `decompose_offset()` (jamais réinjecté dans le tuple
    renvoyé).
  - tout offset non reconnu (ancré comme `'W-MON'`, non position-aware comme
    `'D'`, ou inconnu comme `'foo'`) retombe **silencieusement** sur la
    position par défaut `'E'`, sans erreur ni avertissement — y compris pour
    des fréquences parfaitement valides comme `'D'` qui n'ont simplement pas
    de notion de position.
  - **aucune vérification de type** : une entrée non-string lève une
    `TypeError` **native** de `re` (pas la `ValueError` dédiée de
    `normalize()`) — incohérence entre les méthodes de la même classe.
  - les deux méthodes s'accordent toujours sur la position extraite
    (`extract_position_from_offset(x) == decompose_offset(x)[1]`).
- **`combine_frequency_position()`** :
  - ne s'appuie **pas** sur `_legacy_offset_mapping` (utilisé par
    `decompose_offset()`) mais sur `_position_aware_frequencies`, une liste
    distincte (`['M', 'Q', 'Y', 'W', 'B']`) qui **n'inclut pas `'A'`** —
    asymétrie majeure : `combine_frequency_position('A', 'start')` ignore
    silencieusement la position (`'A'` inchangé), alors que `'Y'`, son
    équivalent non-legacy, produit bien `'YS'`.
  - fréquence non position-aware + position fournie : no-op silencieux,
    **aucun avertissement**.
  - fréquence position-aware + `position=None` : `UserWarning` explicite
    **et** renvoie la fréquence seule (comme un no-op) — comportement
    différent du cas précédent (avertissement vs silence) pour un résultat
    similaire (fréquence inchangée).
  - position invalide (mais fréquence valide) : propage la `ValueError` de
    `normalize()`.
- **`flip_position()`** : accepte code ou littéral en entrée, renvoie
  **toujours un code** en sortie (pas de symétrie de format). Double flip =
  identité. Propage la `ValueError` de `normalize()` sur entrée invalide.
- **Application réaliste** (`df_timeseries`/`df_panel`) : la position d'un
  indicateur donné (ex. `balance_commerciale_annuelle`, publiée en janvier,
  donc en position `'start'` d'année) est indépendante de la fréquence/position
  de l'index global du `DataFrame` qui le contient (ici `MS`) — elle doit être
  déterminée indicateur par indicateur, pas déduite de l'index du tableau.